In [0]:
#Look at the plants data
CATALOG = "capstone_supply_chain"

df = spark.table(f"{CATALOG}.bronze.raw_plant_info")
df.printSchema()
display(df)

In [0]:
#Step 3: Silver plants
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_silver_plants = (spark.table(f"{CATALOG}.bronze.raw_plant_info")
    .select(
        F.col("Plant_Code").alias("plant_code"),
        F.col("Plant_Name").alias("plant_name"),
        F.col("Country").alias("country"),
        F.col("City").alias("city"),
        F.col("Region").alias("region"),
        F.col("Plant_Type").alias("plant_type"),
        F.expr("try_cast(Capacity_Units_Per_Day AS INT)").alias("daily_capacity"),
        F.expr("try_cast(Employees AS INT)").alias("employees"),
        F.col("Warehouse_Code").alias("warehouse_code"),
        F.col("ERP_System").alias("erp_system"),
        F.current_timestamp().alias("silver_processed_at"),
    )
    .filter(F.col("plant_code").isNotNull())
    .dropDuplicates(["plant_code"])
)

df_silver_plants.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.plants")
print("✅ silver.plants rows:", df_silver_plants.count())

In [0]:
#Build the three dimensions
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

# --- dim_vendors ---
df_dim_vendors = (spark.table(f"{CATALOG}.silver.vendors")
    .select(
        "vendor_id", "vendor_name", "country",
        F.col("category").alias("supply_category"),
        "quality_grade", "risk_rating", "risk_category", "payment_terms",
        F.col("certifications_raw").alias("certifications"),
        "num_certifications", "contact_email",
        F.current_timestamp().alias("gold_updated_at"),
    )
)
df_dim_vendors.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_vendors")
print("✅ gold.dim_vendors:", df_dim_vendors.count())

# --- dim_parts ---
df_dim_parts = (spark.table(f"{CATALOG}.silver.inventory")
    .select(
        "part_number", "part_name", "category", "sub_category", "uom",
        "abc_class", "unit_cost_usd", "lead_time_days", "is_hazardous",
        F.current_timestamp().alias("gold_updated_at"),
    )
)
df_dim_parts.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_parts")
print("✅ gold.dim_parts:", df_dim_parts.count())

# --- dim_plants (now from Silver) ---
df_dim_plants = (spark.table(f"{CATALOG}.silver.plants")
    .select(
        "plant_code", "plant_name", "country", "city", "region", "plant_type",
        "daily_capacity", "employees", "warehouse_code", "erp_system",
        F.current_timestamp().alias("gold_updated_at"),
    )
)
df_dim_plants.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_plants")
print("✅ gold.dim_plants:", df_dim_plants.count())

In [0]:
#Check your work
# 1. Is each dimension's key unique and never empty?
for table, key in [("dim_vendors", "vendor_id"), ("dim_parts", "part_number"), ("dim_plants", "plant_code")]:
    d = spark.table(f"{CATALOG}.gold.{table}")
    print(f"{table}: rows={d.count()}, distinct {key}={d.select(key).distinct().count()}, "
          f"null {key}={d.where(F.col(key).isNull()).count()}")

# 2. Does every purchase order point to a vendor, part and plant that exists?
po = spark.table(f"{CATALOG}.silver.purchase_orders")
for fk, table, key in [("vendor_id", "dim_vendors", "vendor_id"),
                       ("part_number", "dim_parts", "part_number"),
                       ("plant_code", "dim_plants", "plant_code")]:
    dim = spark.table(f"{CATALOG}.gold.{table}").select(key)
    missing = po.join(dim, po[fk] == dim[key], "left_anti").select(fk).distinct()
    print(f"POs with a {fk} missing from {table}:", missing.count())
    missing.show()